# PKG edge build: dedup → daily edges → monthly edges → counterparty dimension → entity map

Source `dsihd01p_dsi.neo4j_payments` → target database `bdahd01p_dlcdi1_cdi_tm`. All tables are external parquet tables, partitioned by `month`, and each month is written and registered independently, so a run can resume after an interruption.

| Table | Grain | Built |
|---|---|---|
| `pkg_dedup_5c6c` | one row per 6C duplicate → its 5C twin | one range job for all requested months |
| `pkg_edge_daily` | `(txn_date, src, dst, category)` → amount, n_txn | per month |
| `pkg_edge_monthly` | `(src, dst, category)` per month, + active_days, first/last date | per month, rolled up from daily |
| `pkg_cpty_month` | counterparty × name × FI × type × category per month | per month |
| `pkg_cpty_dim` | one row per counterparty id | rebuilt from all `cpty_month` |
| `pkg_cpty_entity_map` | `cpty_id → ent_l0 / ent_l1 / ent_l2` | rebuilt |

**Rules baked in.**
- **Customers:** `mdm_id` (current). Debit card (8B/8C) is excluded.
- **Counterparty id:**
  - `unq_cpty_id` for 3B/3C (where `unq_cpty_acct_id` is only the originator's 8-digit routing number) and for 7B/9B (name-keyed);
  - `unq_cpty_acct_id` everywhere else;
  - `cpty_id_src` = A or B records which one was used.
- **5C/6C:** the 6C twin is dropped and the 5C row is kept.
- **Self-edges** (same mdm_id on both sides) are kept and flagged `is_self`.
- **`n_non_usd`** counts rows whose currency isn't USD. Their dollars are summed as-is until we confirm `trans_amt` is USD-converted.
- **Entity levels:** `ent_l0` = raw id; `ent_l1` = with the 6C→5C crosswalk applied; `ent_l2` = with business-name merges. Person names are never merged.

**First run: one month (trial).** For the backfill, set `RUN_MONTHS = month_range("2024-01", LAST_COMPLETE)` and rerun. Months already written are skipped unless `FORCE = True`.

In [ ]:
# ---- Config -------------------------------------------------------------------------------
import calendar, datetime as dt
def month_range(a, b):
    y, m = map(int, a.split("-")); y2, m2 = map(int, b.split("-")); out = []
    while (y, m) <= (y2, m2):
        out.append(f"{y:04d}-{m:02d}"); y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return out
_t = dt.date.today().replace(day=1) - dt.timedelta(days=1)
LAST_COMPLETE = f"{_t.year:04d}-{_t.month:02d}"

SRC_TABLE  = "dsihd01p_dsi.neo4j_payments"
TGT_DB     = "bdahd01p_dlcdi1_cdi_tm"
PREFIX     = "pkg_"

RUN_MONTHS    = ["2025-01"]                     # trial.  Backfill: month_range("2024-01", LAST_COMPLETE)
BACKFILL_FROM = "2024-01"                       # only used to extrapolate time and storage
FORCE         = False                           # True = rebuild months already written

EXCLUDE_CODES = {"8B", "8C"}                    # debit card
B_ID_CODES    = {"3B", "3C", "7B", "8B", "8C", "9B"}   # counterparty id = unq_cpty_id here; unq_cpty_acct_id elsewhere
KEEP_CAT, DROP_CAT = "5C.RTP_PRT_CPTY_PAYS", "6C.RTP_P2P_CPTY_PAYS"

XW_MIN_PURITY   = 0.8                           # 6C→5C crosswalk: min share of a 6C id's pairs going to its top 5C id
NAME_KEY_MINLEN = 4                             # ent_l2: shortest normalized business name allowed to merge
FILES = {"edge_daily": 8, "edge_monthly": 4, "cpty_month": 4}   # parquet files per month (~150–300 MB each)
CODEC = "snappy"
SHUFFLE_PARTITIONS = 400
BUILD_DEDUP, BUILD_DIM_MAP, RUN_SENSITIVITY = True, True, True

In [ ]:
# ---- NumPy compat shim (must run before any toPandas) ------------------------------------
import numpy as np, warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"pyspark.*")

from pyspark.sql import SparkSession, Window, functions as F
from IPython.display import display
from urllib.parse import unquote
from contextlib import contextmanager
import pandas as pd, time, os

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
print("Spark", spark.version)
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)

def show(pdf, title, pct=(), money=(), gb=()):
    print(f"\n■ {title}   [{len(pdf):,} rows]")
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                fmt[c] = "{:.2%}"
        elif c in money:                            fmt[c] = "${:,.0f}"
        elif c in gb:                               fmt[c] = "{:,.2f}"
        elif pd.api.types.is_integer_dtype(pdf[c]): fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):   fmt[c] = "{:,.2f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:    sty = sty.hide(axis="index")
    except AttributeError: sty = sty.hide_index()
    display(sty)

def nz(c):
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

TIMINGS = []
@contextmanager
def timed(stage, month=None):
    t0 = time.time(); yield
    s = time.time() - t0
    TIMINGS.append({"month": month, "stage": stage, "seconds": s})
    print(f"  ⏱ {month or '':<8} {stage}: {s/60:,.1f} min")

HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def month_bounds(m):
    y, mo = map(int, m.split("-"))
    return f"{m}-01", f"{m}-{calendar.monthrange(y, mo)[1]:02d}"
code_of = lambda c: c.split(".")[0]
GB = 1e9

## §0 Source categories, target location, table registry

In [ ]:
CATS = set()
for r in spark.sql(f"SHOW PARTITIONS {SRC_TABLE}").collect():
    kv = {k.lower(): unquote(v) for k, v in (p.split("=", 1) for p in r[0].split("/"))}
    CATS.add(kv["category"])
KEEP_CATS = sorted(c for c in CATS if code_of(c) not in EXCLUDE_CODES)
assert KEEP_CAT in CATS and DROP_CAT in CATS, "5C/6C category names not found"
print(f"{len(KEEP_CATS)} categories kept, excluded: {sorted(c for c in CATS if code_of(c) in EXCLUDE_CODES)}")

SRC_COLS = {c.lower() for c in spark.table(SRC_TABLE).columns}
NEED = {"trans_id", "trans_dt", "trans_amt", "category", "mdm_id_pays", "mdm_id_receives", "pnc_dep_acct_receives",
        "unq_cpty_acct_id", "unq_cpty_id", "cpty_name", "cpty_type", "cpty_fin_entity_name", "trans_currency"}
assert not (NEED - SRC_COLS), f"source is missing {NEED - SRC_COLS}"

DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {TGT_DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
KEYS_T = ["dedup_5c6c", "edge_daily", "edge_monthly", "cpty_month", "cpty_dim", "cpty_entity_map"]
T = {k: f"{TGT_DB}.{PREFIX}{k}" for k in KEYS_T}
P = {k: f"{DB_LOC}/{PREFIX}{k}" for k in KEYS_T}
REPL = int(fs_for(DB_LOC).getDefaultReplication(JPath(DB_LOC)))
print("target:", DB_LOC, "| replication", REPL)

def ensure_table(key, schema, partitioned):
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in schema.fields if f.name != "month")
    part = " PARTITIONED BY (month STRING)" if partitioned else ""
    spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {T[key]} ({cols}){part} STORED AS PARQUET LOCATION '{P[key]}'")

def add_partition(key, m):
    spark.sql(f"ALTER TABLE {T[key]} ADD IF NOT EXISTS PARTITION (month='{m}') LOCATION '{P[key]}/month={m}'")

def month_done(key, m):
    return exists(f"{P[key]}/month={m}/_SUCCESS")

def write_month(df, key, m, part_col, sort_cols):
    (df.repartition(FILES[key], part_col).sortWithinPartitions(*sort_cols)
       .write.mode("overwrite").option("compression", CODEC).parquet(f"{P[key]}/month={m}"))
    ensure_table(key, df.schema, True); add_partition(key, m)

def write_full(df, key):
    df.write.mode("overwrite").option("compression", CODEC).parquet(P[key])
    ensure_table(key, df.schema, False); spark.catalog.refreshTable(T[key])

def size_of(path):
    if not exists(path): return 0, 0
    c = fs_for(path).getContentSummary(JPath(path))
    return c.getLength(), c.getFileCount()

## §1 5C/6C dedup map: one range job for every requested month

This is the v2 rule. The key is `(date, receiving account, amount, upper-cased cpty_name)`, with one-to-one pairing and 5C kept. Tier 1 is the same day; tier 2 is 6C stamped one day later. The job reads from one day before the first month so tier-2 pairs at month edges are found. It also stores both counterparty ids, which feed the crosswalk.

In [ ]:
def dedup_pairs(lo, hi):
    lo_ext = (dt.date.fromisoformat(lo) - dt.timedelta(days=1)).isoformat()
    r = (spark.table(SRC_TABLE)
           .filter(F.col("trans_dt").between(lo_ext, hi) & F.col("category").isin(KEEP_CAT, DROP_CAT))
           .select(nz("trans_id").alias("trans_id"), F.to_date("trans_dt").alias("dt"), nz("category").alias("category"),
                   nz("pnc_dep_acct_receives").alias("acct"), F.abs(F.col("trans_amt").cast("decimal(18,2)")).alias("amt"),
                   F.upper(F.regexp_replace(nz("cpty_name"), r"\s+", " ")).alias("nm"), nz("unq_cpty_acct_id").alias("cid"))
           .filter(F.col("acct").isNotNull() & F.col("nm").isNotNull())
           .withColumn("dt0", F.col("dt")))
    A, B = r.filter(F.col("category") == KEEP_CAT), r.filter(F.col("category") == DROP_CAT)
    GK = ["dt", "acct", "amt", "nm"]
    def ranked(df, s):
        w = Window.partitionBy(*GK).orderBy("trans_id")
        return df.withColumn("rn", F.row_number().over(w)).select(
            *GK, "rn", *[F.col(c).alias(f"{c}_{s}") for c in ("trans_id", "dt0", "cid")])
    t1 = ranked(A, "a").join(ranked(B, "b"), GK + ["rn"]).withColumn("tier", F.lit(1))
    A_left = A.join(t1.select(F.col("trans_id_a").alias("trans_id")), "trans_id", "left_anti")
    B_left = B.join(t1.select(F.col("trans_id_b").alias("trans_id")), "trans_id", "left_anti")
    t2 = (ranked(A_left, "a").join(ranked(B_left.withColumn("dt", F.date_sub("dt", 1)), "b"), GK + ["rn"])
            .withColumn("tier", F.lit(2)))
    return (t1.unionByName(t2).filter(F.col("dt0_b") >= F.lit(lo).cast("date"))
              .select(F.col("trans_id_b").alias("trans_id_drop"), F.col("trans_id_a").alias("trans_id_keep"), "tier",
                      F.col("dt0_b").alias("dt_drop"), F.col("dt0_a").alias("dt_keep"), "acct",
                      F.col("amt").alias("amount"), F.col("cid_b").alias("cpty_id_drop"), F.col("cid_a").alias("cpty_id_keep"),
                      F.date_format("dt0_b", "yyyy-MM").alias("month")))

if BUILD_DEDUP:
    todo = [m for m in RUN_MONTHS if FORCE or not exists(f"{P['dedup_5c6c']}/month={m}")]
    if todo:
        lo, hi = month_bounds(min(todo))[0], month_bounds(max(todo))[1]
        with timed(f"dedup range {min(todo)}..{max(todo)} ({len(todo)} months)"):
            pairs = dedup_pairs(lo, hi).filter(F.col("month").isin(todo))
            _prev = spark.conf.get("spark.sql.sources.partitionOverwriteMode", "static")
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")    # replace only these months
            (pairs.repartition("month").write.mode("overwrite").option("compression", CODEC)
                  .partitionBy("month").parquet(P["dedup_5c6c"]))
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", _prev)
            ensure_table("dedup_5c6c", pairs.schema, True)
            for m in todo:
                if exists(f"{P['dedup_5c6c']}/month={m}"): add_partition("dedup_5c6c", m)
    else:
        print("dedup: all requested months already present")
    ds = (spark.read.parquet(P["dedup_5c6c"]).filter(F.col("month").isin(RUN_MONTHS))
               .groupBy("month", "tier").agg(F.count("*").alias("pairs"), F.sum("amount").cast("double").alias("usd"))
               .orderBy("month", "tier").toPandas())
    show(ds, "6C rows flagged as duplicates", money=("usd",))

## §2 Monthly build: daily edges → monthly edges → counterparty-month

In [ ]:
DEDUP_EXISTS = exists(P["dedup_5c6c"])
KEYS = ["txn_date", "src_id", "src_kind", "dst_id", "dst_kind", "category", "cpty_id_src", "is_self"]

def source_rows(m):
    lo, hi = month_bounds(m)
    s = spark.table(SRC_TABLE).filter(F.col("trans_dt").between(lo, hi) & F.col("category").isin(KEEP_CATS))
    code_c = F.split(nz("category"), r"\.").getItem(0)
    use_b  = code_c.isin(list(B_ID_CODES))
    A, B   = nz("unq_cpty_acct_id"), nz("unq_cpty_id")
    cid    = F.when(use_b, F.coalesce(B, A)).otherwise(F.coalesce(A, B))
    cid_src = (F.when(use_b & B.isNotNull(), "B").when(use_b & A.isNotNull(), "A")
                .when(A.isNotNull(), "A").when(B.isNotNull(), "B"))
    ccy = F.upper(nz("trans_currency"))
    rows = s.select(nz("trans_id").alias("trans_id"), F.to_date("trans_dt").alias("txn_date"), nz("category").alias("category"),
                    nz("mdm_id_pays").alias("mp"), nz("mdm_id_receives").alias("mr"),
                    cid.alias("cid"), cid_src.alias("cid_src"),
                    F.abs(F.col("trans_amt").cast("decimal(18,2)")).alias("amt"),
                    (ccy.isNotNull() & (ccy != "USD")).cast("int").alias("non_usd"),
                    nz("cpty_name").alias("cpty_name"), nz("cpty_type").alias("cpty_type"),
                    nz("cpty_fin_entity_name").alias("fi"))
    if DEDUP_EXISTS:
        dd = spark.read.parquet(P["dedup_5c6c"]).filter(F.col("month") == m).select(F.col("trans_id_drop").alias("trans_id"))
        rows = (rows.filter(F.col("category") != DROP_CAT)
                    .unionByName(rows.filter(F.col("category") == DROP_CAT).join(dd, "trans_id", "left_anti")))
    internal = F.col("mp").isNotNull() & F.col("mr").isNotNull()
    return (rows
        .withColumn("src_kind", F.when(F.col("mp").isNotNull(), "CUST").otherwise("CPTY"))
        .withColumn("src_id",   F.when(F.col("mp").isNotNull(), F.col("mp")).otherwise(F.col("cid")))
        .withColumn("dst_kind", F.when(F.col("mr").isNotNull(), "CUST").otherwise("CPTY"))
        .withColumn("dst_id",   F.when(F.col("mr").isNotNull(), F.col("mr")).otherwise(F.col("cid")))
        .withColumn("cpty_id_src", F.when(internal, F.lit(None).cast("string")).otherwise(F.col("cid_src")))
        .withColumn("is_self", internal & (F.col("mp") == F.col("mr"))))

MONTH_LOG = []
for m in RUN_MONTHS:
    if not FORCE and all(month_done(k, m) for k in ("edge_daily", "edge_monthly", "cpty_month")):
        print(f"{m}: already built, skipped"); continue
    t_month = time.time()
    rows = source_rows(m)

    with timed("daily edges: scan + dedup + aggregate + write", m):
        daily = rows.groupBy(*KEYS).agg(F.sum("amt").cast("decimal(18,2)").alias("amount"),
                                        F.count("*").cast("int").alias("n_txn"),
                                        F.sum("non_usd").cast("int").alias("n_non_usd"))
        write_month(daily, "edge_daily", m, "src_id", ["src_id", "dst_id", "txn_date"])

    d = spark.read.parquet(f"{P['edge_daily']}/month={m}")
    with timed("monthly edges: roll-up + write", m):
        monthly = (d.groupBy(*KEYS[1:]).agg(
                      F.sum("amount").cast("decimal(18,2)").alias("amount"), F.sum("n_txn").cast("int").alias("n_txn"),
                      F.sum("n_non_usd").cast("int").alias("n_non_usd"), F.count("*").cast("int").alias("active_days"),
                      F.min("txn_date").alias("first_dt"), F.max("txn_date").alias("last_dt")))
        write_month(monthly, "edge_monthly", m, "src_id", ["src_id", "dst_id"])

    with timed("counterparty-month: scan + aggregate + write", m):
        cm = (rows.filter(F.col("cpty_id_src").isNotNull())
                  .groupBy(F.col("cid").alias("cpty_id"), "cpty_id_src", "category", "cpty_type", "cpty_name", "fi")
                  .agg(F.count("*").alias("n_txn"), F.sum("amt").cast("decimal(22,2)").alias("amount")))
        write_month(cm, "cpty_month", m, "cpty_id", ["cpty_id"])

    with timed("QA: source vs edges reconciliation", m):
        src_t = rows.groupBy("category").agg(F.count("*").alias("src_rows"), F.sum("amt").alias("src_amt"),
                                             F.sum("non_usd").alias("non_usd_rows"),
                                             F.sum(F.col("is_self").cast("int")).alias("self_rows")).toPandas()
        edg_t = d.groupBy("category").agg(F.sum("n_txn").alias("edge_rows"), F.sum("amount").alias("edge_amt"),
                                          F.count("*").alias("daily_edges"),
                                          F.sum((F.col("src_id").isNull() | F.col("dst_id").isNull()).cast("long")).alias("null_id_edges")).toPandas()
        mon_t = spark.read.parquet(f"{P['edge_monthly']}/month={m}").groupBy("category").agg(
                    F.count("*").alias("monthly_edges"), F.sum("amount").alias("mon_amt")).toPandas()
        q = src_t.merge(edg_t, on="category", how="outer").merge(mon_t, on="category", how="outer")
        q["month"] = m
        q["ok"] = (q.src_rows == q.edge_rows) & (q.src_amt == q.edge_amt) & (q.edge_amt == q.mon_amt)
    MONTH_LOG.append(q)
    secs = time.time() - t_month
    with open("pkg_edge_runlog.csv", "a") as f:
        f.write(f"{dt.datetime.now().isoformat(timespec='seconds')},{m},{secs:.0f},{int(q.src_rows.sum())},{bool(q.ok.all())}\n")
    print(f"{m}: done in {secs/60:,.1f} min | reconciliation {'OK' if q.ok.all() else 'MISMATCH ❌'}")

## §3 QA for the months just built

In [ ]:
if MONTH_LOG:
    Q = pd.concat(MONTH_LOG, ignore_index=True)
    Q["code"] = Q.category.map(code_of)
    for c in ("src_amt", "edge_amt", "mon_amt"): Q[c] = Q[c].astype(float)
    Q["daily_vs_rows"]   = Q.daily_edges / Q.src_rows
    Q["monthly_vs_rows"] = Q.monthly_edges / Q.src_rows
    Q["self_share"]      = Q.self_rows / Q.src_rows
    Q["non_usd_share"]   = Q.non_usd_rows / Q.src_rows
    cols = ["month", "code", "src_rows", "src_amt", "daily_edges", "daily_vs_rows", "monthly_edges", "monthly_vs_rows",
            "null_id_edges", "self_share", "non_usd_share", "ok"]
    show(Q[cols].astype({"src_rows": "int64", "daily_edges": "int64", "monthly_edges": "int64", "null_id_edges": "int64"}),
         "Per category: rows → edges, null ids, self-transfers, non-USD, reconciliation",
         pct=("daily_vs_rows", "monthly_vs_rows", "self_share", "non_usd_share"), money=("src_amt",))
    bad = Q[~Q.ok]
    if len(bad):
        show(bad[["month", "code", "src_rows", "edge_rows", "src_amt", "edge_amt", "mon_amt"]], "❌ RECONCILIATION MISMATCH")
        raise AssertionError("row or dollar totals differ between source and edges; see table above")
    print("✓ every category reconciles exactly: source rows = daily n_txn, source $ = daily $ = monthly $")

    ids = (spark.read.parquet(P["edge_monthly"]).filter(F.col("month").isin(RUN_MONTHS))
             .select(F.explode(F.array(
                 F.struct(F.col("src_id").alias("id"), F.col("src_kind").alias("kind")),
                 F.struct(F.col("dst_id").alias("id"), F.col("dst_kind").alias("kind")))).alias("n"))
             .groupBy("n.kind").agg(F.countDistinct("n.id").alias("distinct_nodes")).toPandas())
    show(ids, "Distinct nodes in the monthly edges (all built months)")

## §4 Storage, time, and extrapolation to the backfill

The extrapolation assumes months look like the built ones. Recent volumes vary by ±10% (230–265M non-card rows per month).

In [ ]:
N_BF = len(month_range(BACKFILL_FROM, LAST_COMPLETE))
built = [m for m in RUN_MONTHS if month_done("edge_daily", m)]
st = []
for k in ("dedup_5c6c", "edge_daily", "edge_monthly", "cpty_month"):
    b, f = zip(*[size_of(f"{P[k]}/month={m}") for m in built]) if built else ((0,), (0,))
    per_m = sum(b) / max(len(built), 1)
    st.append({"table": T[k], "GB_per_month": per_m / GB, "files_per_month": sum(f) / max(len(built), 1),
               f"GB_{N_BF}_months": per_m * N_BF / GB, f"GB_on_HDFS_x{REPL}": per_m * N_BF * REPL / GB})
S = pd.DataFrame(st)
S = pd.concat([S, pd.DataFrame([{"table": "TOTAL", **{c: S[c].sum() for c in S.columns if c != "table"}}])], ignore_index=True)
show(S, f"Storage per month and for the backfill ({BACKFILL_FROM} → {LAST_COMPLETE}: {N_BF} months)",
     gb=tuple(c for c in S.columns if c.startswith("GB")))
try:
    q = fs_for(DB_LOC).getQuotaUsage(JPath(DB_LOC))
    if q.getSpaceQuota() >= 0:
        rem = q.getSpaceQuota() - q.getSpaceConsumed()
        print(f"quota at {DB_LOC}: {rem/GB/1000:,.2f} TB remaining (replicated) → backfill uses "
              f"{S.iloc[-1][f'GB_on_HDFS_x{REPL}'] / (rem/GB):.1%}")
except Exception as e:
    print("quota lookup skipped:", str(e)[:120])

TM = pd.DataFrame(TIMINGS)
show(TM.assign(minutes=TM.seconds / 60)[["month", "stage", "minutes"]], "Stage timings")
per_month = TM[TM.month.isin(built)].groupby("month").seconds.sum().mean() if len(TM[TM.month.isin(built)]) else float("nan")
dedup_s   = TM[TM.stage.str.startswith("dedup")].seconds.sum()
dedup_per = dedup_s / max(len(RUN_MONTHS), 1)
show(pd.DataFrame([{"per_month_build_min": per_month / 60, "dedup_per_month_min": dedup_per / 60,
                    f"backfill_{N_BF}_months_hours": (per_month + dedup_per) * N_BF / 3600}]),
     "Time extrapolation (dedup scaled linearly; a single range job usually beats that)")

## §5 Counterparty dimension and entity map (rebuilt from every `cpty_month` partition)

- `ent_l0`: the raw id.
- `ent_l1`: 6C P2P ids folded into their 5C id via the dedup pairs, when purity ≥ `XW_MIN_PURITY`.
- `ent_l2`: `ent_l1`, plus **business-like** names merged on a normalized key (upper-case, letters and digits only, legal suffixes INC/LLC/CO/… removed). Person-like names are never merged.

The table of largest merge groups is there for a sanity check; generic names such as "CITY OF …" show up there first.

In [ ]:
BIZ = (r"\b(INC|INCORPORATED|LLC|L L C|CO|CORP|CORPORATION|COMPANY|LTD|LIMITED|LP|LLP|PC|PLLC|PA|NA|BANK|TRUST|FUND|"
       r"FOUNDATION|ASSOC|ASSOCIATION|SERVICES?|GROUP|HOLDINGS|PARTNERS|UNIVERSITY|COLLEGE|SCHOOL|DISTRICT|CITY|COUNTY|"
       r"STATE|TOWNSHIP|BOROUGH|HOSPITAL|CENTER|CHURCH|DEPT|DEPARTMENT|TREASURER|INSURANCE|SUPPLY|ENTERPRISES|"
       r"INTERNATIONAL|AUTHORITY|SYSTEMS|SOLUTIONS|TECHNOLOGIES|MANAGEMENT|PROPERTIES|REALTY|CONSTRUCTION)\b")
LEGAL = r"\b(INC|INCORPORATED|LLC|L L C|CO|CORP|CORPORATION|COMPANY|LTD|LIMITED|LP|LLP|PC|PLLC|PA|NA|THE)\b"

def name_key(c):
    s = F.regexp_replace(F.upper(c), "[^A-Z0-9 ]", " ")
    s = F.regexp_replace(s, LEGAL, " ")
    s = F.trim(F.regexp_replace(s, r"\s+", " "))
    return F.when(F.length(s) >= NAME_KEY_MINLEN, s)

if BUILD_DIM_MAP:
    with timed("counterparty dimension"):
        cm_all = spark.read.parquet(P["cpty_month"])
        def dominant(col):
            w = Window.partitionBy("cpty_id").orderBy(F.desc("n"), F.col(col))
            return (cm_all.filter(F.col(col).isNotNull()).groupBy("cpty_id", col).agg(F.sum("n_txn").alias("n"))
                          .withColumn("r", F.row_number().over(w)).filter("r = 1").select("cpty_id", col))
        nvar = cm_all.filter(F.col("cpty_name").isNotNull()).groupBy("cpty_id").agg(F.countDistinct("cpty_name").alias("n_name_variants"))
        letter = F.regexp_extract(F.split("category", r"\.").getItem(0), "([ABC])$", 1)
        base = cm_all.groupBy("cpty_id").agg(
                   F.max("cpty_id_src").alias("cpty_id_src"), F.sum("n_txn").alias("n_txn"),
                   F.sum("amount").cast("decimal(22,2)").alias("amount"),
                   F.min("month").alias("first_month"), F.max("month").alias("last_month"),
                   F.countDistinct("month").alias("active_months"),
                   F.concat_ws(",", F.sort_array(F.collect_set(F.split("category", r"\.").getItem(0)))).alias("categories"),
                   F.max((letter == "B").cast("int")).cast("boolean").alias("is_payee"),
                   F.max((letter == "C").cast("int")).cast("boolean").alias("is_payer"))
        dim = (base.join(dominant("cpty_name"), "cpty_id", "left").join(dominant("fi"), "cpty_id", "left")
                   .join(dominant("cpty_type"), "cpty_id", "left").join(nvar, "cpty_id", "left")
                   .withColumn("is_business", F.coalesce(F.upper("cpty_name").rlike(BIZ), F.lit(False)))
                   .withColumn("name_key", name_key(F.col("cpty_name"))))
        write_full(dim, "cpty_dim")

    with timed("entity map"):
        dim = spark.table(T["cpty_dim"])
        xw = (spark.read.parquet(P["dedup_5c6c"]).filter(F.col("cpty_id_drop").isNotNull() & F.col("cpty_id_keep").isNotNull())
                   .groupBy("cpty_id_drop", "cpty_id_keep").agg(F.count("*").alias("pairs")))
        wx = Window.partitionBy("cpty_id_drop").orderBy(F.desc("pairs"), "cpty_id_keep")
        xw = (xw.withColumn("tot", F.sum("pairs").over(Window.partitionBy("cpty_id_drop")))
                .withColumn("r", F.row_number().over(wx)).filter("r = 1")
                .filter(F.col("pairs") / F.col("tot") >= XW_MIN_PURITY)
                .select(F.col("cpty_id_drop").alias("cpty_id"), F.col("cpty_id_keep").alias("xw_to")))
        emap = (dim.select("cpty_id", "cpty_id_src").join(xw, "cpty_id", "left")
                   .withColumn("ent_l0", F.col("cpty_id"))
                   .withColumn("ent_l1", F.coalesce("xw_to", "cpty_id")))
        tgt = dim.select(F.col("cpty_id").alias("ent_l1"), F.col("name_key").alias("t_key"), F.col("is_business").alias("t_biz"))
        emap = (emap.join(tgt, "ent_l1", "left")
                    .withColumn("ent_l2", F.when(F.col("t_biz") & F.col("t_key").isNotNull(),
                                                 F.concat(F.lit("N:"), F.md5("t_key"))).otherwise(F.col("ent_l1")))
                    .withColumn("rule", F.when(F.col("ent_l2").startswith("N:"), "business_name")
                                         .when(F.col("xw_to").isNotNull(), "crosswalk_6c_5c").otherwise("identity"))
                    .select("cpty_id", "cpty_id_src", "ent_l0", "ent_l1", "ent_l2", "rule"))
        write_full(emap, "cpty_entity_map")

    em = spark.table(T["cpty_entity_map"])
    r = em.agg(F.count("*").alias("cpty_ids"), F.countDistinct("ent_l1").alias("entities_l1"),
               F.countDistinct("ent_l2").alias("entities_l2")).toPandas()
    r["l1_vs_ids"] = r.entities_l1 / r.cpty_ids; r["l2_vs_ids"] = r.entities_l2 / r.cpty_ids
    show(r, "Entity consolidation", pct=("l1_vs_ids", "l2_vs_ids"))
    show(em.groupBy("rule", "cpty_id_src").count().orderBy("rule", "cpty_id_src").toPandas(), "Ids by rule and id source")
    big = (em.filter(F.col("rule") == "business_name").groupBy("ent_l2").agg(F.count("*").alias("ids"))
             .orderBy(F.desc("ids")).limit(15)
             .join(spark.table(T["cpty_dim"]).select(F.concat(F.lit("N:"), F.md5("name_key")).alias("ent_l2"), "name_key").distinct(), "ent_l2")
             .select("name_key", "ids").orderBy(F.desc("ids")).toPandas())
    show(big, "Largest business-name merges (sanity check for generic names)")

## §6 Sensitivity: do per-customer counterparty metrics move between `ent_l0` and `ent_l2`?

These are per customer and month, in each direction:
- **Breadth:** the number of distinct counterparties.
- **Concentration:** the top counterparty's share of dollars.

If rankings barely move (high correlation, few large changes), full entity resolution is a refinement rather than a blocker. Churn sensitivity needs two or more consecutive months, so it's computed automatically after the backfill.

In [ ]:
if RUN_SENSITIVITY:
    with timed("sensitivity"):
        em  = spark.table(T["cpty_entity_map"]).select("cpty_id", "ent_l0", "ent_l2")
        mon = spark.read.parquet(P["edge_monthly"]).filter(F.col("month").isin(RUN_MONTHS) & ~F.col("is_self"))
        x = (mon.filter((F.col("src_kind") == "CUST") & (F.col("dst_kind") == "CPTY"))
                .select("month", F.col("src_id").alias("cust"), F.col("dst_id").alias("cpty"), "amount", F.lit("OUT").alias("dir"))
             .unionByName(mon.filter((F.col("src_kind") == "CPTY") & (F.col("dst_kind") == "CUST"))
                .select("month", F.col("dst_id").alias("cust"), F.col("src_id").alias("cpty"), "amount", F.lit("IN").alias("dir")))
             .filter(F.col("cpty").isNotNull())
             .join(em, F.col("cpty") == F.col("cpty_id"), "left")
             .withColumn("l0", F.coalesce("ent_l0", "cpty")).withColumn("l2", F.coalesce("ent_l2", "cpty"))
             .withColumn("amt", F.col("amount").cast("double")))
        def per_cust(lv):
            g = x.groupBy("month", "dir", "cust", lv).agg(F.sum("amt").alias("a"))
            return g.groupBy("month", "dir", "cust").agg(F.count("*").alias(f"n_{lv}"),
                                                         (F.max("a") / F.sum("a")).alias(f"top_{lv}"))
        c = per_cust("l0").join(per_cust("l2"), ["month", "dir", "cust"])
        sens = (c.groupBy("dir").agg(
                    F.count("*").alias("customer_months"),
                    F.expr("percentile_approx(n_l0, 0.5)").alias("median_n_l0"),
                    F.expr("percentile_approx(n_l2, 0.5)").alias("median_n_l2"),
                    F.avg(F.col("n_l2") / F.col("n_l0")).alias("mean_ratio_l2_l0"),
                    F.avg((F.col("n_l2") <= 0.9 * F.col("n_l0")).cast("double")).alias("breadth_down_10pct_plus"),
                    F.avg((F.col("n_l2") <= 0.75 * F.col("n_l0")).cast("double")).alias("breadth_down_25pct_plus"),
                    F.corr(F.log1p("n_l0"), F.log1p("n_l2")).alias("corr_log_breadth"),
                    F.avg((F.abs(F.col("top_l2") - F.col("top_l0")) > 0.05).cast("double")).alias("top_share_moves_5pp_plus"))
                .toPandas())
        show(sens, "ent_l0 vs ent_l2, per customer-month", pct=("mean_ratio_l2_l0", "breadth_down_10pct_plus",
             "breadth_down_25pct_plus", "top_share_moves_5pp_plus"))

### Next
- **Trial looks right?** Set `RUN_MONTHS = month_range("2024-01", LAST_COMPLETE)` and run all cells. Built months are skipped, the dedup covers the whole range in one job, and a stopped run resumes where it left off. `pkg_edge_runlog.csv` next to the notebook logs every month.
- **Impala users** need `REFRESH <table>` after each run.